# Протонный диод В. Лившица: ЭДС ХХ, ток КЗ и ВАХ по схеме Нернста–Планка
*16 ноя 2026. Постановка — по письму В. Лившица от 16.11.2026 и с. 4 материала `Hydrogen_Energetics_NP.pdf`.*

**Схема (одна общая газовая фаза H₂, давление p, температура T):**
H₂(газ) ⇄ 2H(вход, Pt) ⇄ H⁺ + e⁻ → **мембрана** (диффузия ↔ дрейф) → H⁺ + e⁻ ⇄ H(выход, Pt) ⇄ **барьерный слой** ⇄ H₂(тот же газ).

**Как предложил Владимир:**
- ЭДС определяется активностями **атомарного H в каталитических слоях** a_вх и a_вых, а не давлениями газа;
- поток протонов в мембране = диффузия (вход→выход) минус дрейф (обратно, против поля); в ХХ они равны;
- I_КЗ = e·(поток протонов) при эквипотенциальных электродах.

**Единственный неизвестный — барьер.** Вводим параметр κ: во сколько раз абсорбция H₂ барьером (H₂ → 2H) подавлена сильнее, чем требует равновесие H₂ ⇄ 2H.
κ > 1 — гипотеза гомофлюидного осмоса: некаталитический барьер выпускает водород («выход есть»), но почти не впускает молекулы H₂ из газа («входа почти нет»). Ниже: (1) как ЭДС, ток КЗ и ВАХ зависят от κ; (2) количественная оценка доли атомарной десорбции с некаталитического барьера (V, Ni) и рабочей температуры.

Все скорости — закон действующих масс; потенциалы в единицах RT/F; константы — произвольные.

In [ ]:
# preparser(False)  # раскомментировать при запуске в SageMath 9.6

In [ ]:
import numpy as np
RT_F_mV = 8.314462618*300/96485.33212*1e3   # 25.85 мВ при 300 K

def rates(x, P, kappa=1.0, p=1.0):
    """x = [ln a_in, u0, uL, ln a_out, u2]; потенциалы в единицах RT/F; phi1 = 0.
    a_in, a_out — активности атомарного H во входном и выходном каталитических слоях,
    u0, uL — потенциал протонов у входной/выходной границы мембраны (c = 1, Nafion),
    u2 — потенциал выходного электрода. kappa = 1: барьер с микрообратимостью;
    kappa > 1: абсорбция H2 барьером подавлена в kappa раз сильнее, чем «положено» (гипотеза).
    Все потоки — в атомах H (= протонах = электронах) за единицу времени."""
    la, u0, uL, lo, u2 = x
    a, ao = np.exp(la), np.exp(lo)
    r1 = 2*P['k1']*(p - a**2)                                   # H2(газ) ⇄ 2H(вход)
    d1 = u0 - 0.0                                               # перенапряжение входа
    r2 = P['k2']*(a*np.exp(-d1/2) - np.exp(d1/2))               # H(вход) ⇄ H+ + e(эл.1)
    J  = P['g']*(np.exp(u0) - np.exp(uL))                       # мембрана: диффузия + дрейф (Нернст–Планк)
    d2 = uL - u2
    r4 = P['k4']*(np.exp(d2/2) - ao*np.exp(-d2/2))              # H+ + e(эл.2) ⇄ H(выход)
    r5 = 2*P['k5']*(ao**2 - p/kappa)                            # 2H(выход, барьер) ⇄ H2(газ)
    return r1, r2, J, r4, r5

def solve(P, kappa=1.0, mode='load', R=0.0, V=None, x0=None):
    """mode='load': внешняя нагрузка R (R=0 — КЗ, R=inf — ХХ); mode='V': задано V = phi2 - phi1."""
    def F(x):
        r1, r2, J, r4, r5 = rates(x, P, kappa)
        I = r4
        if mode == 'V':   last = x[4] - V
        elif np.isinf(R): last = I                              # ХХ: ток во внешней цепи = 0
        else:             last = x[4] - (-I*R)                  # phi2 = -I*R (ток I течёт 2→1 снаружи)
        return np.array([r1-r2, r2-J, J-r4, r4-r5, last])
    x = np.zeros(5) if x0 is None else np.array(x0, float)
    for _ in range(200):
        f = F(x)
        Jm = np.empty((5,5)); h = 1e-7
        for j in range(5):
            dx = np.zeros(5); dx[j] = h
            Jm[:,j] = (F(x+dx)-f)/h
        step = np.linalg.solve(Jm, -f)
        x += np.clip(step, -2, 2)
        if np.max(abs(step)) < 1e-13: break
    I = rates(x, P, kappa)[3]
    return x, I


## 1. Гипотеза гомофлюидного осмоса: ЭДС и ток КЗ как функции κ

In [ ]:
P = {'k1':1, 'k2':5, 'g':2, 'k4':0.05, 'k5':0.2}
print(' kappa   E_XX, мВ   (RT/2F)ln(kappa)   I_КЗ (отн.)')
for kap in [1, 1.08, 2, 10, 100, 1e4]:
    x, _ = solve(P, kap, 'load', R=np.inf); E = x[4]*RT_F_mV
    _, Isc = solve(P, kap, 'load', R=0.0)
    print(f'{kap:7g}  {E:9.4f}   {RT_F_mV/2*np.log(kap):9.4f}        {Isc:.4f}')

**Результат:** E_ХХ = (RT/2F)·ln κ **точно**, независимо от констант мембраны и электродов. То есть ЭДС измеряет ровно одну величину — асимметрию барьера κ:
- 1 мВ ⇔ κ ≈ 1.08; 10 мВ ⇔ κ ≈ 2.2; 60 мВ ⇔ κ = 100.

I_КЗ, напротив, зависит от кинетики и ограничен самой медленной стадией (здесь k4 — выходной электрод).

Важно: κ ≠ 1 — это допущение, что стадия 2H ⇄ H₂ на барьере имеет отношение прямой и обратной скоростей, отличное от exp(−ΔG/RT). Модель показывает, **что** даст такое допущение, но не обосновывает его.

## 3. ВАХ при внешнем напряжении (потенциостат), κ = 2

In [ ]:
P_asym = {'k1':1, 'k2':5, 'g':2, 'k4':0.05, 'k5':0.2}   # сильно разные электроды
print('  V, мВ      I (отн.)')
for v in np.linspace(-6, 6, 13):
    _, I = solve(P_asym, 2.0, 'V', V=v)
    print(f'{v*RT_F_mV:8.1f}   {I:+.4f}')
_, I0 = solve(P_asym, 2.0, 'V', V=0.0)
x, _ = solve(P_asym, 2.0, 'load', R=np.inf)
print('I(V=0) =', I0, ' = I_КЗ;  V(I=0) =', x[4]*RT_F_mV, 'мВ = E_ХХ = (RT/2F)ln2 =', RT_F_mV/2*np.log(2))

При κ > 1 ВАХ **сдвинута**: проходит не через (0, 0), а через точки I(V=0) = I_КЗ ≠ 0 и V(I=0) = E_ХХ = (RT/2F)·ln κ. Именно эти две точки — признак генерации; асимметрия формы ВАХ (разная кинетика электродов) сама по себе ничего не доказывает.

## 4. Протокол для потенциостата
1. Одна камера H₂ (обе стороны в одном газе), изотермия ΔT < 0.01 K, O₂ < 1 ppm.
2. Измерить V(I=0) электрометром (≥10¹² Ом) часы—сутки и I(V=0) — 0 В на потенциостате.
3. Снять ВАХ ±200 мВ; проверить, проходит ли через (0, 0).
4. Перевернуть ячейку: эффект барьера обязан сменить знак.
5. Контроль: та же ячейка без барьерного слоя (или с барьером с обеих сторон) — эффект должен исчезнуть.
6. Калибровка: разделить камеры, p_вых/p_вх = 1.01 → ожидается 0.13 мВ (Нернст).

**Пересчёт:** измеренная E_ХХ = ε мВ ⇒ κ = exp(2ε/25.85). Отличие κ от 1 выше погрешности (с контролями 4–5) было бы прямым экспериментальным свидетельством нарушения микрообратимости на барьере.

## 2. Доля атомарной десорбции с некаталитического барьера (V, Ni) и рабочая температура
**Постановка (по с. 4 и 8 материала).** Двухслойный выходной электрод: внутренний слой Pd (к мембране) выделяет H без перенапряжения; внешний барьерный слой (V или Ni), нанесённый внахлёст, обращён к газу и каталитическими свойствами не обладает. Значит, на внешней поверхности рекомбинация 2H → H₂ подавлена, и водород может уходить в газ **атомами** — «наиболее горячими», преодолевшими работу выхода H из металла. Процесс эндотермичен и задаёт рабочую температуру.

**Что нужно оценить:** долю f атомов H среди частиц, уходящих с барьера, и теплоту, уносимую одним атомом.

**Термодинамика выхода атома [данные + память].** Энергия выхода атома из металла в газ
Q_H = ½D(H₂) − ΔH_s,  где ½D(H₂) = 218 кДж/моль H [данные, справочник], ΔH_s — теплота растворения H в металле [память, уточнить]:
- V: ΔH_s ≈ −30 кДж/моль H (экзотермично) ⇒ Q_H ≈ 248 кДж/моль ≈ 2.57 эВ;
- Ni: ΔH_s ≈ +16 кДж/моль H (эндотермично) ⇒ Q_H ≈ 202 кДж/моль ≈ 2.09 эВ.
Для сравнения: молекулярная десорбция уносит на атом лишь ≈ −ΔH_s (+30 и −16 кДж/моль), то есть в 10 раз меньше.

**Доля атомов f [оценка].** Если растворённый H в барьере в равновесии с газом H₂ при p (режим ХХ), то давление атомарного H, равновесное с металлом, равно газовому равновесию p_H = √(K_p·p), **независимо от металла** (металл входит в обе ветви одинаково через свою растворимость и сокращается). Различия между V и Ni остаются только в коэффициентах прилипания:
f = j_H / (2 j_H₂) ≈ (√2/2)·(α_H/α_H₂)·√(K_p/p),
α_H ~ 0.1–0.3 [данные, Hatano 2007 и др.: 0.1–0.4 для металлов с неметаллическим монослоем], α_H₂ ~ 10⁻⁴…10⁻² для некаталитической поверхности [данные: 2·10⁻⁴ для V(111)+O/S, Eibl–Winkler 2002; для чистого Ni выше — уточнить].
В режиме генерации выходной электрод «перекачивает» H в барьер выше равновесия (активность a > 1): атомный поток растёт ∝ a, молекулярный ∝ a², так что f ∝ 1/a — **ещё меньше**.

In [ ]:
R = 8.314462618
def Kp(T):   # H2 = 2H, бар; ΔH°298 = 436.0 кДж, ΔS°298 = 98.75 Дж/К, ΔCp ≈ const
    dH0, dS0, dCp = 435998., 2*114.717 - 130.680, 2*20.786 - 29.0
    return np.exp(-((dH0 + dCp*(T-298.15)) - T*(dS0 + dCp*np.log(T/298.15)))/(R*T))
print('проверка: log10 Kp(1000 K) =', round(np.log10(Kp(1000)),2), '(JANAF: -17.29)')
def f_at(T, p=1.0, aH=0.2, aH2=1e-3): return (np.sqrt(2)/2)*(aH/aH2)*np.sqrt(Kp(T)/p)
print('\n T,K   f (α_H2=1e-2)   f (α_H2=2e-4)    при p = 1 бар; при p = 1 мбар f больше в 30 раз')
for T in (400, 500, 600, 800, 1000, 1200, 1500):
    print(f'{T:5d}   {f_at(T,aH2=1e-2):11.1e}    {f_at(T,aH2=2e-4):11.1e}')
# температура, при которой f = 1e-3 и 1e-2
for ftar in (1e-3, 1e-2):
    for aH2 in (1e-2, 2e-4):
        lo, hi = 300., 3000.
        for _ in range(100):
            m = 0.5*(lo+hi); lo, hi = (m, hi) if f_at(m, aH2=aH2) < ftar else (lo, m)
        print(f'f = {ftar:g} при α_H2 = {aH2:g}:  T ≈ {lo:.0f} K')
# Связь с κ (оценка): если атомный канал односторонний (атомы уходят, обратно атомами не приходят),
# асимметрия барьера не больше κ_max ≈ 1 + f  =>  E_XX ≤ (RT/2F)·ln(1+f)
for T in (600, 1000, 1500):
    f = f_at(T, aH2=2e-4); x, _ = solve(P, 1+f, 'load', R=np.inf)
    E = x[4]*RT_F_mV*T/300
    print(f'T={T} K: f={f:.1e}  E_XX ≤ {E:.2e} мВ;  проверка (RT/2F)ln(1+f) = {RT_F_mV*T/300/2*np.log(1+f):.2e} мВ')

**Результат [оценка].**
1. До ~800 K атомарная десорбция ничтожна: f ≲ 10⁻⁷…10⁻⁹. Заметной (f ~ 10⁻³…10⁻²) она становится только при **≈1300–1900 K** (при p = 1 бар; при 1 мбар — примерно на 200 K ниже). Это выше рабочего диапазона любых протонных проводников (Nafion ≤ 90 °C, CsH₂PO₄ ≈ 240 °C, BaZrY ≤ 600–700 °C).
2. **Различие V и Ni** — только через отношение α_H/α_H₂; теплота растворения в f сокращается. Энергетически атом с V уносит ≈ 2.6 эВ, с Ni ≈ 2.1 эВ — это и есть «испарительное» охлаждение барьера, но на один атом, а их доля мала.
3. Если атомный канал рассматривать как источник асимметрии κ, то κ − 1 ≲ f, и ЭДС ≲ (RT/2F)·f — при 1000 K ≲10⁻⁴ мВ, при 1500 K ≲1 мВ. Для 1 мВ нужно κ ≈ 1.08, то есть f ~ 0.08 — недостижимо в рабочем диапазоне мембран.
4. **Что это значит для опыта.** Если ЭДС в милливольты будет измерена при 300–600 K, объяснить её атомарной десорбцией с барьера (в рамках этой оценки) нельзя; механизм асимметрии должен быть другим — именно тогда результат и интересен. Оценка опирается на ΔH_s и α по памяти/частичным данным; коэффициенты прилипания H₂ к чистым V и Ni нужно уточнить по первоисточникам.